# Load New York City SHR records

This notebook appends New York City SHR source records from the master `SHR65_23.csv` file to the shared SQLite `shr` table.

NYC is defined by the five counties/boroughs: Bronx, Kings, New York, Queens, and Richmond. The NYT analytical incident scope is 1981 through 2000, inclusive.

In [1]:
import pandas as pd
import sqlite3

In [2]:
shr_file = "SHR65_23.csv"
shr = pd.read_csv(shr_file)
shr.shape

(894636, 30)

In [3]:
nyc_counties = {
    "Bronx, NY",
    "Kings, NY",
    "New York, NY",
    "Queens, NY",
    "Richmond, NY",
}

shr_nyc = shr[
    shr["CNTYFIPS"].isin(nyc_counties)
    & shr["Year"].between(1981, 2000)
].copy()
shr_nyc

,ID,CNTYFIPS,Ori,State,Agency,Agentype,Source,Solved,Year,Month,...,OffRace,OffEthnic,Weapon,Relationship,Circumstance,Subcircum,VicCount,OffCount,FileDate,MSA
584931,198101003NY03030,"New York, NY",NY03030,New York,New York,Municipal police,FBI,Yes,1981,January,...,White,Unknown or not reported,Knife or cutting instrument,Friend,Circumstances undetermined,NaN,0,0,32982.0,"New York-New Jersey-Long Island, NY-NJ"
584932,198101011NY03030,"New York, NY",NY03030,New York,New York,Municipal police,FBI,Yes,1981,January,...,Black,Unknown or not reported,Knife or cutting instrument,Stranger,Other arguments,NaN,0,0,32982.0,"New York-New Jersey-Long Island, NY-NJ"
584933,198101028NY03030,"New York, NY",NY03030,New York,New York,Municipal police,FBI,Yes,1981,January,...,Black,Unknown or not reported,Knife or cutting instrument,Relationship not determined,Burglary,NaN,0,0,32982.0,"New York-New Jersey-Long Island, NY-NJ"
584934,198101035NY03030,"New York, NY",NY03030,New York,New York,Municipal police,FBI,Yes,1981,January,...,Black,Unknown or not reported,Knife or cutting instrument,Boyfriend,Other arguments,NaN,0,0,32982.0,"New York-New Jersey-Long Island, NY-NJ"
584935,198101038NY03030,"New York, NY",NY03030,New York,New York,Municipal police,FBI,Yes,1981,January,...,Black,Unknown or not reported,"Handgun - pistol, revolver, etc",Boyfriend,Circumstances undetermined,NaN,0,0,32982.0,"New York-New Jersey-Long Island, NY-NJ"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632364,198203001NY540RL,"Queens, NY",NY540RL,New York,Queens Long Island Railroad,Special police,FBI,No,1982,March,...,Unknown,Unknown or not reported,"Handgun - pistol, revolver, etc",Relationship not determined,Circumstances undetermined,NaN,0,0,100682.0,"New York-New Jersey-Long Island, NY-NJ"
632365,198802001NY540RL,"Queens, NY",NY540RL,New York,Queens Long Island Railroad,Special police,FBI,No,1988,February,...,Unknown,Unknown or not reported,"Blunt object - hammer, club, etc",Relationship not determined,Circumstances undetermined,NaN,0,0,32289.0,"New York-New Jersey-Long Island, NY-NJ"
632366,198907001NY540RL,"Queens, NY",NY540RL,New York,Queens Long Island Railroad,Special police,FBI,Yes,1989,July,...,Unknown,Unknown or not reported,Other or type unknown,Friend,Other,NaN,0,0,31390.0,"New York-New Jersey-Long Island, NY-NJ"
632367,199008001NY540RL,"Queens, NY",NY540RL,New York,Queens Long Island Railroad,Special police,FBI,No,1990,August,...,Unknown,Unknown or not reported,Other or type unknown,Relationship not determined,All suspected felony type,NaN,0,0,21691.0,"New York-New Jersey-Long Island, NY-NJ"


In [4]:
# Coverage and agency audit
shr_nyc.groupby("CNTYFIPS").size().sort_index()

CNTYFIPS
Bronx, NY           1
Kings, NY           1
New York, NY    29276
Queens, NY          6
dtype: int64

In [5]:
shr_nyc.groupby("Year").size().sort_index()

Year
1981    1574
1982    1662
1983    1758
1984    1560
1985    1301
1986    1397
1987    1601
1988    1840
1989    1646
1990    1937
1991    2083
1992    2089
1993    2072
1994    1655
1995    1242
1996    1021
1997     793
1998     668
1999     688
2000     697
dtype: int64

In [6]:
shr_nyc.groupby(["CNTYFIPS", "Agency", "Agentype", "Ori"]).size().sort_values(ascending=False)

CNTYFIPS      Agency                                   Agentype          Ori    
New York, NY  New York                                 Municipal police  NY03030    29263
              Long Island RailRoad - Nassau County     Special police    NY529RL        9
Queens, NY    Queens Long Island Railroad              Special police    NY540RL        6
New York, NY  Long Island Rail Road - New York County  Special police    NY530RL        4
Bronx, NY     State Park: Bronx County                 Special police    NY462PB        1
Kings, NY     Kings Long Island Railroad               Special police    NY523RL        1
dtype: int64

In [8]:
dates = pd.to_datetime([f"{m}-1-{y}" for m, y in zip(shr_nyc["Month"], shr_nyc["Year"])])
date_periods = [
    pd.Period(freq="m", year=y, month=m)
    for m, y in zip(dates.month, dates.year)
]
shr_nyc["YearMonth"] = date_periods
shr_nyc["YearMonth"] = shr_nyc["YearMonth"].astype(str)
shr_nyc[["Year", "Month", "YearMonth"]].head()

/var/folders/rd/cfj58vmx6wl_y1cx_dsn7lq40000gn/T/ipykernel_19905/4040734778.py:3: FutureWarning: 'm' is deprecated and will be removed in a future version, please use 'M' instead.
  pd.Period(freq="m", year=y, month=m)


,Year,Month,YearMonth
584931,1981,January,1981-01
584932,1981,January,1981-01
584933,1981,January,1981-01
584934,1981,January,1981-01
584935,1981,January,1981-01


In [9]:
con = sqlite3.connect("newarticles.db")
shr_nyc.to_sql("shr", if_exists="append", con=con, index=True)
con.close()